# Training API

Fine-tune forecasting models on your Lightning Rod datasets. This notebook walks through the full training workflow: generating a dataset, estimating cost, creating a training job, and monitoring progress.

The training API supports LoRA fine-tuning with configurable base models, training steps, batch size, and rank.

## Install the SDK

In [1]:
%pip install lightningrod-ai python-dotenv openai

from IPython.display import clear_output
clear_output()

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Prepare the dataset

Training requires a dataset ID from a pipeline run. Run one of the other notebooks first to generate a dataset - each one prints the **Dataset ID** after `transforms.run()` — copy it into the cell below.

In [3]:
dataset_id = config.get_config_value("LIGHTNINGROD_DATASET_ID")

dataset = lr.datasets.get(dataset_id)
_ = dataset.download()


In [4]:
from lightningrod import prepare_for_training, FilterParams, SplitParams

train_dataset, test_dataset = prepare_for_training(
    dataset,
    filter=FilterParams(days_to_resolution_range=(90, None)),
    split=SplitParams(test_size=0.2),
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> prepare_for_training                                                                                        │
│                                                                                                                 │
│    Starting with 435 samples                                                                                    │
│                                                                                                                 │
│    Filter:  Dropped 53 invalid, 245 horizon → 137 remain                                                        │
│    Dedup:   137 remain (0 duplicates)                                                                           │
│    Split:   Splits: 34 train | 28 test (0 dropped, no prediction_date)                                          │
│             75 train samples removed for leakage                                                                │
│                                                                                                                 │
│  ⚠ Unhealthy dataset                                                                                            │
│                                                                                                                 │
│  75/109 train samples (68%) were removed for temporal leakage — the date_close or resolution_date of train      │
│  questions extends into the test period.                                                                        │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Extend the seed generator (date) filter range to start earlier — questions generated near the start      │
│  will resolve well before the test window. Aim for at least 2× your max resolution horizon.                     │
│      • Generate more samples by increasing max_seeds in lr.transforms.run() or removing the limit, or increase  │
│  questions_per_seed in your question generator config. A larger, temporally well-spread dataset naturally       │
│  pushes the split cutoff far enough back.                                                                       │
│      • If very few seeds were returned by the pipeline (check the run summary table), the search queries may    │
│  not surface results across the full date range. Try more diverse search queries, increase                      │
│  articles_per_search, or shorten interval_duration_days.                                                        │
│                                                                                                                 │
│  Only 34 train samples remain after preparation. This is below the recommended minimum of +1000 for effective   │
│  training.                                                                                                      │
│                                                                                                                 │
│    Tips:                                                                                                        │
│      • Increase max_seeds in lr.transforms.run() to generate more samples.                                      │
│      • Increase questions_per_seed in your question generator (ForwardLookingQuestionGenerator or               │
│  QuestionGenerator) to produce more questions from each seed article.Add more search queries to your seed       │
│  generator to diversify seed sources.                                                                           │
│      • Widen the seed generator date range (start_date

## Estimate training cost

Before starting a job, use `estimate_cost` to see the expected cost and token usage.

In [5]:
from lightningrod import GRPOTrainingConfig

config = GRPOTrainingConfig(
    base_model_id="openai/gpt-oss-120b",
    training_steps=50,
)
cost_estimate = lr.training.estimate_cost(config, dataset=train_dataset)
print(f"Estimated cost: ${cost_estimate.total_cost_dollars:.2f}")
print(f"Effective steps: {cost_estimate.effective_steps}")
print(f"Train tokens: {cost_estimate.train_tokens:,}")
print(f"Notes: {cost_estimate.notes}")

Estimated cost: $0.11
Effective steps: 2
Train tokens: 151,348
Notes: Estimate uses per-answer-type output token estimates; actual may vary


## Start training

`run` creates a job and polls until completion with a live progress display.


In [6]:
training_job = lr.training.run(config, dataset=train_dataset, name="Forecasting fine-tune")
print(f"Job {training_job.id} completed with status: {training_job.status}")
print(f"Trained model ID: {training_job.model_id}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Training COMPLETED                                                                                          │
│                                                                                                                 │
│    Job ID: c1977fc5-c547-472a-b11b-5799b5f40b9d                                                                 │
│                                                                                                                 │
│    Model:                                                                                                       │
│  checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk3OWRiOWRlMjRkOnRyYWluOjAvc2FtcGxlcl93ZWlnaHRzL3N0ZXBfMDAwMg      │
│                                                                                                                 │
│    reward: latest -0.0210  avg -0.0711  (2 steps)  (higher is better)                                           │
│        ▁█                                                                                                       │
│    mean_output_tokens: latest 503.6250  avg 513.9727  (2 steps)                                                 │
│        █▁                                                                                                       │
│                                                                                                                 │
│    Cost:  $0.17                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Job c1977fc5-c547-472a-b11b-5799b5f40b9d completed with status: COMPLETED
Trained model ID: checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk3OWRiOWRlMjRkOnRyYWluOjAvc2FtcGxlcl93ZWlnaHRzL3N0ZXBfMDAwMg


## Inference with your trained model

Use `lr.predict()` to run inference with your trained model. You can also use the OpenAI-compatible API directly — see [08_foresight_model.ipynb](08_foresight_model.ipynb) for the pre-trained foresight model.


In [7]:
print(lr.predict(training_job.model_id, "Will the Fed cut rates by 25bp in March 2026?"))


**Short answer:**  
At the time of writing (early May 2026) the consensus among most market participants and most analysts is that a **25‑basis‑point (bp) rate cut in March 2026 is possible but not highly likely**. The probability implied by fed‑funds futures and the CME FedWatch Tool hovers around **30‑40 %**, with the balance of the curve still pricing for a *pause* or a *gradual easing* that would begin later in the year (June‑July) if inflation continues to ease.

Below is a detailed walk‑through of the factors that drive that outlook, the data‑driven signals the Fed has been sending, the market pricing, and the key “what‑if” scenarios that could swing the odds either way.

---

## 1. Current monetary‑policy backdrop (as of May 2026)

| Indicator | Latest reading (April 2026) | Target / Trend |
|-----------|----------------------------|----------------|
| **Effective federal funds rate** | 4.75 % (unchanged since Jan 2026) | Highest level since 2008 |
| **Core PCE inflation (12‑mo 

## Run evals on trained model

Run test evals on your trained model against a test dataset. The eval job runs the model on the dataset, reports metrics, and can include a reasoning comparison between the base and fine-tuned model. Use the same dataset for a quick check, or a separate test split for production.

In [8]:
from lightningrod import training

eval_job = lr.evals.run_from_training_job(
    config,
    training_job,
    test_dataset,
    reasoning_comparison_sample_size=20,
)

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Eval COMPLETED                                                                                              │
│                                                                                                                 │
│    Job ID: 125c9c0f-4b20-4397-bfc1-54e3fbb79129                                                                 │
│    Dataset: ad98afcb-da82-4f7b-ada5-5447122e992c                                                                │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┓                                                                 │
│  ┃ Metric              ┃    Base ┃ Fine-tuned ┃                                                                 │
│  ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━┩                                                                 │
│  │ brier_score         │  0.1859 │     0.1793 │                                                                 │
│  │ ece                 │  0.1618 │     0.1379 │                                                                 │
│  │ mc_ece              │       — │          — │                                                                 │
│  │ mean_reward         │ -0.1859 │    -0.1793 │                                                                 │
│  │ mean_valid_reward   │ -0.1859 │    -0.1793 │                                                                 │
│  │ n_samples           │      28 │         28 │                                                                 │
│  │ n_valid             │      28 │         28 │                                                                 │
│  │ parse_rate          │  1.0000 │     1.0000 │                                                                 │
│  │ total_cost          │  0.0120 │     0.0120 │                                                                 │
│  │ total_input_tokens  │   28924 │      28924 │                                                                 │
│  │ total_output_tokens │   15502 │      15435 │                                                                 │
│  └─────────────────────┴─────────┴────────────┘                                                                 │
│                                                                                                                 │
│  Reasoning comparison analysis                                                                                  │
│  # Global Reasoning Quality Report: openai/gpt-oss-120b vs. checkpoint:MjU1Z...                                 │
│                                                                                                                 │
│  ## 1) Overall Verdict                                                                                          │
│                                                                                                                 │
│  Both models demonstrate broadly similar reasoning patterns — evidence-grounded probability estimation with     │
│  explicit factor enumeration and calibrated uncertainty language. **openai/gpt-oss-120b** tends to produce      │
│  slightly higher probability estimates and more exploratory, self-correcting deliberation, while the            │
│  **checkpoint** model is more conservative and crisper, occasionally sacrificing depth for brevity. Neither     │
│  model shows systematic factual errors, but both share identical structural weaknesses: thin base-rate          │
│  grounding, limited use of reference classes, and occasional factual slips (e.g., Storm Shadow status with      │
│  Ukraine). The differences in final estimates are mode

In [9]:
results = lr.evals.load_results(eval_job.id)

display(results[training_job.config.base_model_id].head())
display(results[training_job.model_id].head())

,sample_id,prompt,output,parsed_answer,correct_answer,reward,reasoning,answer_type,input_tokens,output_tokens,cost_dollars,model_id
0,7111cf80-a8ff-4c45-bc9f-2cf4d156928f,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.75</answer>,0.75,1,-0.0625,We need to estimate probability that US Africa...,binary,1082,332,0.000341,openai/gpt-oss-120b
1,dcd72507-26b4-43f4-8384-57a395a7b848,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.58</answer>,0.58,0,-0.3364,We need to estimate probability that US milita...,binary,1190,485,0.000428,openai/gpt-oss-120b
2,41a3f4fc-1f41-45e5-b763-d097672244d3,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.12</answer>,0.12,0,-0.0144,We need to estimate probability that US Marine...,binary,442,544,0.000319,openai/gpt-oss-120b
3,62518159-c040-4c66-aada-b6c77e344ed6,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.95</answer>,0.95,1,-0.0025,We need to assess probability that MCADT will ...,binary,457,186,0.000164,openai/gpt-oss-120b
4,9edebedd-623e-4af4-9d46-1efbae3c3618,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.15</answer>,0.15,0,-0.0225,We need to estimate probability that IDF will ...,binary,1020,463,0.000387,openai/gpt-oss-120b


,sample_id,prompt,output,parsed_answer,correct_answer,reward,reasoning,answer_type,input_tokens,output_tokens,cost_dollars,model_id
0,7111cf80-a8ff-4c45-bc9f-2cf4d156928f,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.78</answer>,0.78,1,-0.0484,We need to estimate probability that US Africa...,binary,1082,421,0.000380,checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk...
1,dcd72507-26b4-43f4-8384-57a395a7b848,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.35</answer>,0.35,0,-0.1225,We need to assess probability that US military...,binary,1190,545,0.000454,checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk...
2,41a3f4fc-1f41-45e5-b763-d097672244d3,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.07</answer>,0.07,0,-0.0049,We need to estimate probability that US Marine...,binary,442,574,0.000332,checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk...
3,62518159-c040-4c66-aada-b6c77e344ed6,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.88</answer>,0.88,1,-0.0144,We need to assess probability that MCADT will ...,binary,457,300,0.000214,checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk...
4,9edebedd-623e-4af4-9d46-1efbae3c3618,<|start|>system<|message|>Knowledge cutoff: 20...,<answer>0.30</answer>,0.3,0,-0.0900,We need to estimate probability that IDF will ...,binary,1020,541,0.000422,checkpoint:MjU1ZjgyNWItMTVkYy01ODFlLThmZmItODk...


> Note: the trained model checkpoint will only be available for the period of 7 days. If you wish to host this model long-term, reach out to us at support@lightningrod.ai.